In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/nabilatabassummumu/our-dataset/merged_dataset.csv
/kaggle/input/datasets/nabilatabassummumu/batd-dataset/merge_dataset_aust.csv


In [44]:
!pip install --upgrade transformers

In [26]:
!pip install -q transformers datasets sentencepiece scikit-learn

# import pandas as pd
# import numpy as np
# import torch
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
# from datasets import Dataset
# from transformers import (
#     AutoTokenizer,
#     AutoModelForSequenceClassification,
#     Trainer,
#     TrainingArguments,
#     DataCollatorWithPadding
# )

import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, DataCollatorWithPadding, TrainingArguments
from datasets import Dataset

In [3]:
os.listdir("/kaggle/input/datasets/nabilatabassummumu/our-dataset")

['merged_dataset.csv']

In [4]:
df = pd.read_csv("/kaggle/input/datasets/nabilatabassummumu/our-dataset/merged_dataset.csv")

# print(df.head())
print(df["category"].value_counts())

category
politics                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   3420
sports                                                                                                                                                                                                                         

In [5]:
# Count category frequencies
category_counts = df["category"].value_counts()

# Keep only categories with more than 2 samples
valid_categories = category_counts[category_counts > 2].index

# Filter dataframe
df = df[df["category"].isin(valid_categories)]

# Optional: check result
print(df["category"].value_counts())

category
politics         3420
sports           2695
international    2300
national         1359
literature       1251
economy           861
economics         416
education          57
bangladesh         44
business           19
Name: count, dtype: int64


In [6]:
print(df["source"].value_counts())

source
gpt                    3681
claud                  2301
Samakal                1462
pdf                    1203
Prothom Alo            1141
Bangladesh Pratidin     943
Ittefaq                 820
Daily Star Bangla       463
Kalbela                 248
Dhaka Post              149
Jugantor                 11
Name: count, dtype: int64


In [7]:
print(df["class"].value_counts())

class
1    6440
0    5982
Name: count, dtype: int64


In [8]:
# checking for NaN value
df["category"].isna().sum()

np.int64(0)

In [11]:
# ---- Step 2: Create combined stratification column ----
df["stratify_col"] = df["source"].astype(str) + "_" + df["category"].astype(str)

# ---- Step 3: Remove rare combinations (important!) ----
combo_counts = df["stratify_col"].value_counts()
valid_combos = combo_counts[combo_counts > 1].index
df = df[df["stratify_col"].isin(valid_combos)]

# ---- Step 4: Stratified split ----
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["stratify_col"]
)

# ---- Optional: Drop helper column ----
train_df = train_df.drop(columns=["stratify_col"])
test_df = test_df.drop(columns=["stratify_col"])

# ---- Check distribution ----
print(train_df["category"].value_counts(normalize=True))
print(test_df["category"].value_counts(normalize=True))

category
politics         0.275463
sports           0.216888
international    0.184984
national         0.109400
literature       0.100644
economy          0.069444
economics        0.033514
education        0.004630
bangladesh       0.003523
business         0.001510
Name: proportion, dtype: float64
category
politics         0.274849
sports           0.217304
international    0.185513
national         0.109457
literature       0.101006
economy          0.068813
economics        0.033400
education        0.004427
bangladesh       0.003622
business         0.001610
Name: proportion, dtype: float64


In [12]:
print(test_df.columns)

Index(['source', 'category', 'data', 'class'], dtype='object')


In [13]:
# Keep only necessary columns
# train_df = train_df[["text", "label"]]
# test_df = test_df[["text", "label"]]

# Keep only necessary columns
train_df = train_df[["data", "class"]]
test_df = test_df[["data", "class"]]

train_df = train_df.rename(columns={"data": "text", "class": "label"})
test_df = test_df.rename(columns={"data": "text", "class": "label"})

### Convert to HuggingFace Dataset

In [14]:
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

### Load BanglaBert

In [15]:
model_name = "csebuetnlp/banglabert"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

### Tokenization

In [16]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/9936 [00:00<?, ? examples/s]

Map:   0%|          | 0/2485 [00:00<?, ? examples/s]

### Set Format for PyTorch

In [19]:
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

### Load Model

In [20]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoi

### Define Evaluation Metrics

In [22]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    accuracy = accuracy_score(labels, preds)
    precision = precision_score(labels, preds)
    recall = recall_score(labels, preds)
    f1 = f1_score(labels, preds)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

### Training Arguments

In [23]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


### Trainer

In [27]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

### Train

In [28]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.448920,0.242332,0.968612,0.969744,0.969744,0.969744
2,0.200043,0.213900,0.974245,0.964367,0.986811,0.975460
3,0.132258,0.230552,0.975453,0.981947,0.970520,0.976200


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

TrainOutput(global_step=1863, training_loss=0.22203243437852516, metrics={'train_runtime': 990.2693, 'train_samples_per_second': 30.101, 'train_steps_per_second': 1.881, 'total_flos': 3921407169085440.0, 'train_loss': 0.22203243437852516, 'epoch': 3.0})

### Evaluate

In [29]:
results = trainer.evaluate()
print(results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.21242353320121765, 'eval_accuracy': 0.9750503018108652, 'eval_precision': 0.9658314350797267, 'eval_recall': 0.9868114817688131, 'eval_f1': 0.9762087490406753, 'eval_runtime': 26.8088, 'eval_samples_per_second': 92.694, 'eval_steps_per_second': 5.819, 'epoch': 3.0}


### Predict

In [31]:
predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(axis=1)

test_df["predicted"] = preds

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


### K-fold validation

In [1]:
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding 
from datasets import Dataset
import numpy as np

# Load dataset
df = pd.read_csv("/kaggle/input/datasets/nabilatabassummumu/our-dataset/merged_dataset.csv")

# Rename columns
df = df.rename(columns={"data": "text", "class": "label"})

# ---- Step 1: Remove rare categories ----
cat_counts = df["category"].value_counts()
df = df[df["category"].isin(cat_counts[cat_counts > 2].index)]

# ---- Step 2: Create combined stratification ----
df["stratify_col"] = df["source"].astype(str) + "_" + df["category"].astype(str) + "_" + df["label"].astype(str)

# ---- Step 3: Remove rare combinations ----
combo_counts = df["stratify_col"].value_counts()
df = df[df["stratify_col"].isin(combo_counts[combo_counts >= 5].index)]

# Load tokenizer
model_name = "csebuetnlp/banglabert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=256)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

# Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df["text"], df["label"])):
    print(f"\n===== Fold {fold+1} =====")

    train_df = df.iloc[train_idx][["text", "label"]]
    val_df = df.iloc[val_idx][["text", "label"]]
    
    train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
    val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

    train_dataset = train_dataset.map(tokenize_function, batched=True)
    val_dataset = val_dataset.map(tokenize_function, batched=True)

    train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )

    training_args = TrainingArguments(
        output_dir=f"./results_fold_{fold}",
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.01,
        disable_tqdm=False
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    results = trainer.evaluate()

    print(results)
    all_results.append(results)

# Average results
avg_results = {
    metric: np.mean([fold_result[f"eval_{metric}"] for fold_result in all_results])
    for metric in ["accuracy", "precision", "recall", "f1"]
}

print("\n===== Final Averaged Results =====")
print(avg_results)

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]


===== Fold 1 =====


...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoi

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.404654,0.232472,0.971003,0.961977,0.982906,0.972329
2,0.153352,0.168598,0.983085,0.989772,0.977467,0.983581
3,0.057609,0.159229,0.984696,0.987510,0.982906,0.985202


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.15922942757606506, 'eval_accuracy': 0.9846959323399114, 'eval_precision': 0.9875097580015613, 'eval_recall': 0.9829059829059829, 'eval_f1': 0.985202492211838, 'eval_runtime': 27.4354, 'eval_samples_per_second': 90.503, 'eval_steps_per_second': 5.686, 'epoch': 3.0}

===== Final Averaged Results =====
{'accuracy': np.float64(0.9786622138431836), 'precision': np.float64(0.9841523279410467), 'recall': np.float64(0.9745224582181103), 'f1': np.float64(0.9793062729233497)}
